# Fundamentos de Ingeniería de Datos
## ALMACENAMIENTO: prueba de carga y consulta

Guardamos **los mismos datos** de dos formas y medimos **carga**, **disco** y **consulta**:

| Forma | Estrategia |
|---|---|
| Archivo Parquet (DuckDB lo consulta directo) | Data lake / lakehouse |
| Tabla en una base DuckDB | Data warehouse |

La prueba se hace para los **tres datasets**. Cada uno tiene su sección con los mismos pasos; lo único que cambia es su celda de configuración (Parquet, `MULTIPLICAR` y consultas).

| Alias | Etiqueta | Filas reales | `MULTIPLICAR` | Filas de prueba |
|---|---|---|---|---|
| `df1` | `vg_ventas_ratings` | 16,713 | 100 | 1,671,300 |
| `df2` | `vg_global_codificado` | 7,086 | 200 | 1,417,200 |
| `df3` | `vg_engagement` | 1,115 | 1,000 | 1,115,000 |

`MULTIPLICAR` se eligió para que los tres queden entre 1.1 y 1.7 millones de filas: con los datos reales los tiempos son de milésimas y no se alcanzan a distinguir.

### Configuración del entorno

In [ ]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import PROYECTO_BASE, PROCESSED_PATH, SERVING_DB, PRUEBAS_PATH, MODELADO_REPORTES_PATH, tamano_mb

print()
print('utils.py importado correctamente')
print()

In [ ]:
# ── Librerías y carpetas ───────────────────────────────────────────────────

import os
import time
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)


# Destino: archivos de prueba y reportes de mediciones (rutas en utils.py)
os.makedirs(PRUEBAS_PATH, exist_ok=True)
os.makedirs(MODELADO_REPORTES_PATH, exist_ok=True)

print()
print(f'Origen   : {PROCESSED_PATH}')
print(f'Serving  : {SERVING_DB}')
print(f'Pruebas  : {PRUEBAS_PATH}')
print(f'Reportes : {MODELADO_REPORTES_PATH}')
print()

---
## Funciones de la prueba

Cada función es un paso de la prueba. Así los tres datasets se miden **exactamente igual** y solo cambia la configuración.

In [ ]:
# ── Funciones de la prueba ─────────────────────────────────────────────────

def crear_datos_prueba(parquet, multiplicar):
    """PASO 1: lee el Parquet y repite sus filas 'multiplicar' veces."""
    con_mem = duckdb.connect()                                    # DuckDB en memoria, solo para preparar
    df = con_mem.sql(f'''
        SELECT b.*
        FROM read_parquet('{parquet}') AS b, range({multiplicar})
    ''').df()
    con_mem.close()
    return df


def cargar_parquet(df, ruta):
    """PASO 2a: guarda el DataFrame como archivo Parquet (data lake). Regresa los segundos."""
    inicio = time.perf_counter()
    df.to_parquet(ruta, index=False)
    return time.perf_counter() - inicio


def cargar_duckdb(df, ruta):
    """PASO 2b: guarda el DataFrame como tabla 'datos' en una base DuckDB (data warehouse).
    Regresa los segundos y la conexión abierta, que se usa en los pasos 3 y 4."""
    if os.path.exists(ruta):
        os.remove(ruta)                                           # empezar de cero en cada ejecución
    con = duckdb.connect(ruta)
    con.register('df_prueba', df)                                 # DuckDB ve el DataFrame como una tabla
    inicio = time.perf_counter()
    con.execute('CREATE TABLE datos AS SELECT * FROM df_prueba')
    segundos = time.perf_counter() - inicio
    con.unregister('df_prueba')
    return segundos, con


def medir_consulta(con, sql):
    """PASOS 4 y 5: ejecuta una consulta y regresa el resultado y los segundos."""
    inicio = time.perf_counter()
    resultado = con.sql(sql).df()
    return resultado, time.perf_counter() - inicio


def tabla_mediciones(m):
    """PASO 6: arma la tabla de mediciones de un dataset a partir del diccionario m."""
    return pd.DataFrame([
        {'destino': 'Parquet', 'estrategia': 'Data lake / lakehouse', 'multiplicar': m['multiplicar'],
         'filas': m['filas_prueba'], 'carga_seg': round(m['carga_parquet'], 2),
         'disco_mb': round(m['mb_parquet'], 1), 'consulta_seg': round(m['consulta_parquet'], 3)},

        {'destino': 'DuckDB', 'estrategia': 'Data warehouse', 'multiplicar': m['multiplicar'],
         'filas': m['filas_prueba'], 'carga_seg': round(m['carga_duckdb'], 2),
         'disco_mb': round(m['mb_duckdb'], 1), 'consulta_seg': round(m['consulta_duckdb'], 3)},

        # serving.db guarda los tres esquemas con datos reales: aquí solo se mide la consulta
        {'destino': 'serving.db', 'estrategia': 'Esquema estrella (datos reales)', 'multiplicar': 1,
         'filas': m['filas_estrella'], 'carga_seg': None,
         'disco_mb': None, 'consulta_seg': round(m['consulta_estrella'], 3)},
    ])


print()
print('Funciones listas')
print()

---
# `df1` — `vg_ventas_ratings`

Consulta de la prueba: **ventas globales por género** (la misma en Parquet, en DuckDB y en el esquema estrella).

In [ ]:
# ── Configuración df1 ─────────────────────────────────────────────────────────

ETIQUETA    = 'vg_ventas_ratings'
PARQUET     = f'{PROCESSED_PATH}/transformed_vg_ventas_ratings.parquet'
MULTIPLICAR = 100

PRUEBA_PARQUET = f'{PRUEBAS_PATH}/prueba_{ETIQUETA}.parquet'            # data lake
PRUEBA_DUCKDB  = f'{PRUEBAS_PATH}/prueba_{ETIQUETA}_duckdb.db'          # data warehouse
MEDICIONES_CSV = f'{MODELADO_REPORTES_PATH}/mediciones_{ETIQUETA}.csv'

# Misma consulta para los dos destinos: {origen} se reemplaza por el Parquet o por la tabla
CONSULTA = '''
    SELECT genre, ROUND(SUM(global_sales), 2) AS total
    FROM {origen}
    GROUP BY genre
    ORDER BY total DESC
'''

# La misma pregunta sobre el esquema estrella en serving.db (datos reales)
CONSULTA_ESTRELLA = '''
    SELECT j.genero, ROUND(SUM(f.ventas_global), 2) AS total
    FROM fact_ventas_df1 AS f
    JOIN dim_juego_df1   AS j USING (juego_id)
    GROUP BY j.genero
    ORDER BY total DESC
'''
TABLA_HECHOS = 'fact_ventas_df1'

m = {'multiplicar': MULTIPLICAR}                                    # aquí se juntan las mediciones de df1

print()
print(f'Dataset : {ETIQUETA}')
print(f'Origen  : {PARQUET}')
print()

### PASO 1 — Datos de prueba (df1)

In [ ]:
# ── Leer el Parquet y multiplicar las filas ────────────────────────────────

df_prueba = crear_datos_prueba(PARQUET, MULTIPLICAR)
m['filas_prueba'] = len(df_prueba)

print()
print(f'Filas originales x {MULTIPLICAR} = {len(df_prueba):,} filas de prueba')
print()

### PASO 2 — ¿Cuánto tarda en guardarse? (df1)

In [ ]:
# ── Carga al data lake (Parquet) y al data warehouse (DuckDB) ──────────────

m['carga_parquet']           = cargar_parquet(df_prueba, PRUEBA_PARQUET)
m['carga_duckdb'], con_duck  = cargar_duckdb(df_prueba, PRUEBA_DUCKDB)

print()
print(f"Parquet : {m['carga_parquet']:.2f} segundos")
print(f"DuckDB  : {m['carga_duckdb']:.2f} segundos")
print()

### PASO 3 — ¿Cuánto ocupa? (df1)

In [ ]:
# ── Tamaño de cada archivo en MB ───────────────────────────────────────────

con_duck.execute('CHECKPOINT')                                    # termina de escribir en el archivo

m['mb_parquet'] = tamano_mb(PRUEBA_PARQUET)
m['mb_duckdb']  = tamano_mb(PRUEBA_DUCKDB)

print()
print(f"Parquet : {m['mb_parquet']:8.1f} MB")
print(f"DuckDB  : {m['mb_duckdb']:8.1f} MB")
print()

### PASO 4 — ¿Cuánto tarda en responder? (df1)

La misma consulta en los dos. Los totales deben coincidir.

In [ ]:
# ── Consulta en el lakehouse (Parquet) y en el data warehouse (tabla DuckDB) ─

res_parquet, m['consulta_parquet'] = medir_consulta(con_duck, CONSULTA.format(origen=f"read_parquet('{PRUEBA_PARQUET}')"))
res_duckdb,  m['consulta_duckdb']  = medir_consulta(con_duck, CONSULTA.format(origen='datos'))

print()
print(f"Parquet : {m['consulta_parquet']:.3f} segundos")
print(f"DuckDB  : {m['consulta_duckdb']:.3f} segundos")
print()

# Medir solo vale si las dos dan el mismo resultado
print(f"Total Parquet : {res_parquet['total'].sum():,.2f}")
print(f"Total DuckDB  : {res_duckdb['total'].sum():,.2f}")
print(f"¿Iguales?     : {res_parquet.equals(res_duckdb)}")
print()

res_duckdb.head()

### PASO 5 — Consulta sobre el esquema estrella (df1)

`serving.db` con los datos reales (sin multiplicar), en solo lectura.

In [ ]:
# ── Consulta con JOIN sobre el esquema estrella ────────────────────────────

con_serving = duckdb.connect(SERVING_DB, read_only=True)          # solo lectura: no modifica el modelo

res_estrella, m['consulta_estrella'] = medir_consulta(con_serving, CONSULTA_ESTRELLA)
m['filas_estrella'] = con_serving.sql(f'SELECT COUNT(*) FROM {TABLA_HECHOS}').fetchall()[0][0]

con_serving.close()

print()
print(f"Esquema estrella : {m['consulta_estrella']:.3f} segundos ({m['filas_estrella']:,} filas en {TABLA_HECHOS})")
print(f"Total            : {res_estrella['total'].sum():,.2f} (ventas (millones), datos reales)")
print()

res_estrella.head()

### PASO 6 — Tabla de mediciones (df1)

In [ ]:
# ── Guardar la tabla de mediciones y liberar memoria ───────────────────────

con_duck.close()
del df_prueba                                                     # los datos de prueba ya no se usan

mediciones_df1 = tabla_mediciones(m)
mediciones_df1.to_csv(MEDICIONES_CSV, index=False)

print()
print(f'Mediciones guardadas en: {MEDICIONES_CSV}')
print()

mediciones_df1

---
# `df2` — `vg_global_codificado`

Consulta de la prueba: **ventas globales por género** (la misma en Parquet, en DuckDB y en el esquema estrella).

In [ ]:
# ── Configuración df2 ─────────────────────────────────────────────────────────

ETIQUETA    = 'vg_global_codificado'
PARQUET     = f'{PROCESSED_PATH}/transformed_vg_global_codificado.parquet'
MULTIPLICAR = 200

PRUEBA_PARQUET = f'{PRUEBAS_PATH}/prueba_{ETIQUETA}.parquet'            # data lake
PRUEBA_DUCKDB  = f'{PRUEBAS_PATH}/prueba_{ETIQUETA}_duckdb.db'          # data warehouse
MEDICIONES_CSV = f'{MODELADO_REPORTES_PATH}/mediciones_{ETIQUETA}.csv'

# Misma consulta para los dos destinos: {origen} se reemplaza por el Parquet o por la tabla
CONSULTA = '''
    SELECT genero, ROUND(SUM(global_sales), 2) AS total
    FROM {origen}
    GROUP BY genero
    ORDER BY total DESC
'''

# La misma pregunta sobre el esquema estrella en serving.db (datos reales)
CONSULTA_ESTRELLA = '''
    SELECT g.genero, ROUND(SUM(f.ventas_global), 2) AS total
    FROM fact_ventas_df2 AS f
    JOIN dim_genero_df2  AS g USING (genero_id)
    GROUP BY g.genero
    ORDER BY total DESC
'''
TABLA_HECHOS = 'fact_ventas_df2'

m = {'multiplicar': MULTIPLICAR}                                    # aquí se juntan las mediciones de df2

print()
print(f'Dataset : {ETIQUETA}')
print(f'Origen  : {PARQUET}')
print()

### PASO 1 — Datos de prueba (df2)

In [ ]:
# ── Leer el Parquet y multiplicar las filas ────────────────────────────────

df_prueba = crear_datos_prueba(PARQUET, MULTIPLICAR)
m['filas_prueba'] = len(df_prueba)

print()
print(f'Filas originales x {MULTIPLICAR} = {len(df_prueba):,} filas de prueba')
print()

### PASO 2 — ¿Cuánto tarda en guardarse? (df2)

In [ ]:
# ── Carga al data lake (Parquet) y al data warehouse (DuckDB) ──────────────

m['carga_parquet']           = cargar_parquet(df_prueba, PRUEBA_PARQUET)
m['carga_duckdb'], con_duck  = cargar_duckdb(df_prueba, PRUEBA_DUCKDB)

print()
print(f"Parquet : {m['carga_parquet']:.2f} segundos")
print(f"DuckDB  : {m['carga_duckdb']:.2f} segundos")
print()

### PASO 3 — ¿Cuánto ocupa? (df2)

In [ ]:
# ── Tamaño de cada archivo en MB ───────────────────────────────────────────

con_duck.execute('CHECKPOINT')                                    # termina de escribir en el archivo

m['mb_parquet'] = tamano_mb(PRUEBA_PARQUET)
m['mb_duckdb']  = tamano_mb(PRUEBA_DUCKDB)

print()
print(f"Parquet : {m['mb_parquet']:8.1f} MB")
print(f"DuckDB  : {m['mb_duckdb']:8.1f} MB")
print()

### PASO 4 — ¿Cuánto tarda en responder? (df2)

La misma consulta en los dos. Los totales deben coincidir.

In [ ]:
# ── Consulta en el lakehouse (Parquet) y en el data warehouse (tabla DuckDB) ─

res_parquet, m['consulta_parquet'] = medir_consulta(con_duck, CONSULTA.format(origen=f"read_parquet('{PRUEBA_PARQUET}')"))
res_duckdb,  m['consulta_duckdb']  = medir_consulta(con_duck, CONSULTA.format(origen='datos'))

print()
print(f"Parquet : {m['consulta_parquet']:.3f} segundos")
print(f"DuckDB  : {m['consulta_duckdb']:.3f} segundos")
print()

# Medir solo vale si las dos dan el mismo resultado
print(f"Total Parquet : {res_parquet['total'].sum():,.2f}")
print(f"Total DuckDB  : {res_duckdb['total'].sum():,.2f}")
print(f"¿Iguales?     : {res_parquet.equals(res_duckdb)}")
print()

res_duckdb.head()

### PASO 5 — Consulta sobre el esquema estrella (df2)

`serving.db` con los datos reales (sin multiplicar), en solo lectura.

In [ ]:
# ── Consulta con JOIN sobre el esquema estrella ────────────────────────────

con_serving = duckdb.connect(SERVING_DB, read_only=True)          # solo lectura: no modifica el modelo

res_estrella, m['consulta_estrella'] = medir_consulta(con_serving, CONSULTA_ESTRELLA)
m['filas_estrella'] = con_serving.sql(f'SELECT COUNT(*) FROM {TABLA_HECHOS}').fetchall()[0][0]

con_serving.close()

print()
print(f"Esquema estrella : {m['consulta_estrella']:.3f} segundos ({m['filas_estrella']:,} filas en {TABLA_HECHOS})")
print(f"Total            : {res_estrella['total'].sum():,.2f} (ventas (millones), datos reales)")
print()

res_estrella.head()

### PASO 6 — Tabla de mediciones (df2)

In [ ]:
# ── Guardar la tabla de mediciones y liberar memoria ───────────────────────

con_duck.close()
del df_prueba                                                     # los datos de prueba ya no se usan

mediciones_df2 = tabla_mediciones(m)
mediciones_df2.to_csv(MEDICIONES_CSV, index=False)

print()
print(f'Mediciones guardadas en: {MEDICIONES_CSV}')
print()

mediciones_df2

---
# `df3` — `vg_engagement`

Consulta de la prueba: **jugadores por género principal** (la misma en Parquet, en DuckDB y en el esquema estrella).

In [ ]:
# ── Configuración df3 ─────────────────────────────────────────────────────────

ETIQUETA    = 'vg_engagement'
PARQUET     = f'{PROCESSED_PATH}/transformed_raw_vg_engagement.parquet'
MULTIPLICAR = 1000

PRUEBA_PARQUET = f'{PRUEBAS_PATH}/prueba_{ETIQUETA}.parquet'            # data lake
PRUEBA_DUCKDB  = f'{PRUEBAS_PATH}/prueba_{ETIQUETA}_duckdb.db'          # data warehouse
MEDICIONES_CSV = f'{MODELADO_REPORTES_PATH}/mediciones_{ETIQUETA}.csv'

# Misma consulta para los dos destinos: {origen} se reemplaza por el Parquet o por la tabla
CONSULTA = '''
    SELECT genero_principal, SUM(plays) AS total
    FROM {origen}
    GROUP BY genero_principal
    ORDER BY total DESC
'''

# La misma pregunta sobre el esquema estrella en serving.db (datos reales)
CONSULTA_ESTRELLA = '''
    SELECT g.genero, SUM(f.jugadores) AS total
    FROM fact_engagement_df3 AS f
    JOIN dim_genero_df3      AS g USING (genero_id)
    GROUP BY g.genero
    ORDER BY total DESC
'''
TABLA_HECHOS = 'fact_engagement_df3'

m = {'multiplicar': MULTIPLICAR}                                    # aquí se juntan las mediciones de df3

print()
print(f'Dataset : {ETIQUETA}')
print(f'Origen  : {PARQUET}')
print()

### PASO 1 — Datos de prueba (df3)

In [ ]:
# ── Leer el Parquet y multiplicar las filas ────────────────────────────────

df_prueba = crear_datos_prueba(PARQUET, MULTIPLICAR)
m['filas_prueba'] = len(df_prueba)

print()
print(f'Filas originales x {MULTIPLICAR} = {len(df_prueba):,} filas de prueba')
print()

### PASO 2 — ¿Cuánto tarda en guardarse? (df3)

In [ ]:
# ── Carga al data lake (Parquet) y al data warehouse (DuckDB) ──────────────

m['carga_parquet']           = cargar_parquet(df_prueba, PRUEBA_PARQUET)
m['carga_duckdb'], con_duck  = cargar_duckdb(df_prueba, PRUEBA_DUCKDB)

print()
print(f"Parquet : {m['carga_parquet']:.2f} segundos")
print(f"DuckDB  : {m['carga_duckdb']:.2f} segundos")
print()

### PASO 3 — ¿Cuánto ocupa? (df3)

In [ ]:
# ── Tamaño de cada archivo en MB ───────────────────────────────────────────

con_duck.execute('CHECKPOINT')                                    # termina de escribir en el archivo

m['mb_parquet'] = tamano_mb(PRUEBA_PARQUET)
m['mb_duckdb']  = tamano_mb(PRUEBA_DUCKDB)

print()
print(f"Parquet : {m['mb_parquet']:8.1f} MB")
print(f"DuckDB  : {m['mb_duckdb']:8.1f} MB")
print()

### PASO 4 — ¿Cuánto tarda en responder? (df3)

La misma consulta en los dos. Los totales deben coincidir.

In [ ]:
# ── Consulta en el lakehouse (Parquet) y en el data warehouse (tabla DuckDB) ─

res_parquet, m['consulta_parquet'] = medir_consulta(con_duck, CONSULTA.format(origen=f"read_parquet('{PRUEBA_PARQUET}')"))
res_duckdb,  m['consulta_duckdb']  = medir_consulta(con_duck, CONSULTA.format(origen='datos'))

print()
print(f"Parquet : {m['consulta_parquet']:.3f} segundos")
print(f"DuckDB  : {m['consulta_duckdb']:.3f} segundos")
print()

# Medir solo vale si las dos dan el mismo resultado
print(f"Total Parquet : {res_parquet['total'].sum():,.2f}")
print(f"Total DuckDB  : {res_duckdb['total'].sum():,.2f}")
print(f"¿Iguales?     : {res_parquet.equals(res_duckdb)}")
print()

res_duckdb.head()

### PASO 5 — Consulta sobre el esquema estrella (df3)

`serving.db` con los datos reales (sin multiplicar), en solo lectura.

In [ ]:
# ── Consulta con JOIN sobre el esquema estrella ────────────────────────────

con_serving = duckdb.connect(SERVING_DB, read_only=True)          # solo lectura: no modifica el modelo

res_estrella, m['consulta_estrella'] = medir_consulta(con_serving, CONSULTA_ESTRELLA)
m['filas_estrella'] = con_serving.sql(f'SELECT COUNT(*) FROM {TABLA_HECHOS}').fetchall()[0][0]

con_serving.close()

print()
print(f"Esquema estrella : {m['consulta_estrella']:.3f} segundos ({m['filas_estrella']:,} filas en {TABLA_HECHOS})")
print(f"Total            : {res_estrella['total'].sum():,.2f} (jugadores, datos reales)")
print()

res_estrella.head()

### PASO 6 — Tabla de mediciones (df3)

In [ ]:
# ── Guardar la tabla de mediciones y liberar memoria ───────────────────────

con_duck.close()
del df_prueba                                                     # los datos de prueba ya no se usan

mediciones_df3 = tabla_mediciones(m)
mediciones_df3.to_csv(MEDICIONES_CSV, index=False)

print()
print(f'Mediciones guardadas en: {MEDICIONES_CSV}')
print()

mediciones_df3

---
# Resumen de los tres datasets

In [ ]:
# ── Las tres tablas de mediciones juntas ───────────────────────────────────

resumen = pd.concat({'vg_ventas_ratings'   : mediciones_df1,
                     'vg_global_codificado': mediciones_df2,
                     'vg_engagement'       : mediciones_df3},
                    names=['dataset', None]).reset_index(level=0)

resumen

---
## Cómo leer los resultados

| Medición | Qué favorece al Parquet | Qué favorece a DuckDB |
|---|---|---|
| Carga | Escribe un solo archivo columnar, sin índices ni catálogo | Guarda los datos ya listos dentro de la base |
| Disco | Comprime cada columna (diccionarios + compresión) | Reserva bloques y guarda metadatos de la base |
| Consulta | DuckDB lee solo las columnas que pide la consulta | Los datos ya están en el formato interno de la base, no hay que leer el archivo |

La interpretación con nuestros números está en la sección 6 del reporte.

> La carpeta `Pruebas/` se puede borrar al terminar. Se conservan los tres `Reportes/mediciones_<etiqueta>.csv`.